In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text('environment', 'dev')
dbutils.widgets.text('catalog_name', 'naturapet_dev')
dbutils.widgets.text('storage_account', 'demodldb')
dbutils.widgets.text('storage_container', 'democodex')
dbutils.widgets.text('data_year', '2026')
dbutils.widgets.text('domain_name', 'comercial')

environment = dbutils.widgets.get('environment').strip().lower()
catalog = dbutils.widgets.get('catalog_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
storage_container = dbutils.widgets.get('storage_container').strip()
data_year = dbutils.widgets.get('data_year').strip()
domain = dbutils.widgets.get('domain_name').strip().lower()
silver_schema = f'{domain}_silver'
storage_uri = f'abfss://{storage_container}@{storage_account}.dfs.core.windows.net'
silver_root = f'{storage_uri}/external/{environment}/{domain}/silver/{data_year}'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`')

def safe_divide(numerator, denominator):
    return F.when(denominator.isNull() | (denominator == 0), F.lit(None)).otherwise(numerator / denominator)

def table_exists(table_name):
    return spark.catalog.tableExists(f'{catalog}.{silver_schema}.{table_name}')

def read_table(table_name):
    return spark.table(f'`{catalog}`.`{silver_schema}`.`{table_name}`')

def write_silver(df, table_name):
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    df = df.withColumn('_np_silver_step', F.lit('business_derivations')).withColumn('_np_silver_processed_ts', F.current_timestamp())
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').option('path', target_path).saveAsTable(target_table))
    return {'table_name': table_name, 'rows': df.count(), 'target_path': target_path}

def derive(table_name, df):
    if table_name == 'dim_producto':
        df = df.withColumn('margen_lista_pct', safe_divide(F.col('precio_lista') - F.col('costo_estandar'), F.col('precio_lista')))
    elif table_name == 'fact_ventas_cabecera':
        df = (df.withColumn('base_neta_sin_iva', F.col('valor_bruto') - F.col('descuento_total'))
                .withColumn('margen_pct', safe_divide(F.col('margen_bruto'), F.col('base_neta_sin_iva')))
                .withColumn('descuento_pct_real', safe_divide(F.col('descuento_total'), F.col('valor_bruto')))
                .withColumn('ticket_lineas_promedio', safe_divide(F.col('valor_total'), F.col('numero_lineas')))
                .withColumn('tiene_campana', F.col('campana_id').isNotNull()))
    elif table_name == 'fact_ventas_detalle':
        df = (df.withColumn('precio_neto_unitario_sin_iva', safe_divide(F.col('base_sin_iva'), F.col('cantidad')))
                .withColumn('precio_unitario_con_iva', safe_divide(F.col('valor_total_linea'), F.col('cantidad')))
                .withColumn('margen_linea_pct', safe_divide(F.col('margen_bruto_linea'), F.col('base_sin_iva')))
                .withColumn('descuento_aplicado', F.col('descuento_valor') > 0))
    elif table_name == 'fact_devoluciones':
        df = (df.withColumn('devolucion_finalizada', F.col('estado_devolucion').isin('Aprobada', 'Rechazada'))
                .withColumn('devolucion_en_revision', F.col('estado_devolucion') == F.lit('En revision')))
        if table_exists('fact_ventas_cabecera'):
            ventas = read_table('fact_ventas_cabecera').select('venta_id', F.col('fecha_venta').alias('_fecha_venta_origen'))
            df = (df.join(ventas, 'venta_id', 'left')
                    .withColumn('dias_hasta_devolucion', F.datediff(F.col('fecha_devolucion'), F.col('_fecha_venta_origen')))
                    .drop('_fecha_venta_origen'))
    elif table_name == 'fact_marketing':
        df = (df.withColumn('ctr', safe_divide(F.col('clics'), F.col('impresiones')))
                .withColumn('conversion_rate', safe_divide(F.col('conversiones_atribuidas'), F.col('leads')))
                .withColumn('cpc', safe_divide(F.col('gasto_marketing'), F.col('clics')))
                .withColumn('cpl', safe_divide(F.col('gasto_marketing'), F.col('leads')))
                .withColumn('roas_calculado', safe_divide(F.col('ventas_atribuidas_valor'), F.col('gasto_marketing'))))
    elif table_name == 'fact_inventario_mensual':
        df = (df.withColumn('stock_promedio', (F.col('stock_inicial') + F.col('stock_final')) / F.lit(2))
                .withColumn('alerta_reorden', F.col('stock_final') <= F.col('punto_reorden'))
                .withColumn('alerta_stock_seguridad', F.col('stock_final') <= F.col('stock_seguridad'))
                .withColumn('rotacion_estimada', safe_divide(F.col('salidas_ventas'), F.col('stock_promedio'))))
    elif table_name == 'fact_ajustes_inventario':
        df = df.withColumn('impacto_merma', F.col('merma_unidades') > 0)
    elif table_name == 'fact_compras':
        df = (df.withColumn('costo_compra_promedio_sin_iva', safe_divide(F.col('subtotal_sin_iva'), F.col('cantidad_comprada')))
                .withColumn('costo_compra_promedio_con_iva', safe_divide(F.col('valor_total_compra'), F.col('cantidad_comprada')))
                .withColumn('iva_pct_real', safe_divide(F.col('iva_valor'), F.col('subtotal_sin_iva')))
                .withColumn('orden_recibida', F.col('estado_orden') == F.lit('Recibida'))
                .withColumn('orden_abierta', F.col('estado_orden').isin('Pendiente', 'Parcial')))
    elif table_name == 'fact_logistica_entregas':
        df = (df.withColumn('entrega_a_tiempo', (F.col('estado_entrega') == F.lit('Entregado')) & F.col('fecha_entrega').isNotNull() & (F.col('dias_retraso') <= 0))
                .withColumn('entrega_tardia', (F.col('estado_entrega') == F.lit('Entregado tarde')) | (F.col('fecha_entrega').isNotNull() & (F.col('dias_retraso') > 0)))
                .withColumn('entrega_fallida', F.col('estado_entrega') == F.lit('Intento fallido'))
                .withColumn('entrega_cancelada', F.col('estado_entrega') == F.lit('Cancelado en ruta'))
                .withColumn('entrega_pendiente', F.col('fecha_entrega').isNull())
                .withColumn('tipo_entrega_operativa', F.when(F.col('empleado_repartidor_id').isNotNull(), F.lit('Interna')).otherwise(F.lit('Tercero')))
                .withColumn('costo_por_kg', safe_divide(F.col('costo_envio'), F.col('peso_estimado_kg'))))
    elif table_name == 'fact_costos_mensual':
        df = (df.withColumn('variacion_costo', F.col('monto_real') - F.col('monto_presupuestado'))
                .withColumn('variacion_costo_pct', safe_divide(F.col('variacion_costo'), F.col('monto_presupuestado'))))
    elif table_name == 'fact_presupuesto_mensual':
        df = df.withColumn('ejecucion_presupuesto_pct', safe_divide(F.col('monto_ejecutado'), F.col('monto_presupuesto')))
    return df

summary = []
tables = [row.tableName for row in spark.sql(f'SHOW TABLES IN `{catalog}`.`{silver_schema}`').collect() if not row.isTemporary and row.tableName != 'silver_quality_checks']
for table_name in tables:
    df = derive(table_name, read_table(table_name))
    summary.append(write_silver(df, table_name))

display(spark.createDataFrame(summary))
